# 4. Train the autoencoder

Runs the same noise -> encode -> decode -> cross-entropy loop as `train.py`, step by step, with a loss curve plotted at the end.

**Scope note:** the full `ModelConfig()` (~1.04B params) needs real GPU memory and RAM (AdamW's optimizer state alone is ~16GB at fp32). If you're on a laptop or CPU-only box, use `DEMO_CONFIG` below to exercise the exact same code path at a scale that actually fits; switch to `ModelConfig()` once you have the hardware for a real run.

In [ ]:
import sys, pathlib, random
sys.path.append(str(pathlib.Path.cwd().parent))

import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
from tokenizers import ByteLevelBPETokenizer

from model.config import ModelConfig
from model.autoencoder import TransformerAutoencoder
from noise import corrupt
from train import encode_batch, shift_right, load_lines

In [ ]:
USE_FULL_SIZE_MODEL = False  # flip to True on a machine with a GPU / plenty of RAM

DEMO_CONFIG = dict(d_model=256, n_heads=8, n_encoder_layers=4,
                    n_decoder_layers=4, d_ff=1024, max_seq_len=64)

cfg = ModelConfig() if USE_FULL_SIZE_MODEL else ModelConfig(**DEMO_CONFIG)

corpus_path = "../data/corpus.txt"
vocab_dir = "../tokenizer/vocab"
assert pathlib.Path(corpus_path).exists(), "Run 02_generate_corpus.ipynb first"
assert pathlib.Path(f"{vocab_dir}/vocab.json").exists(), "Run 03_train_tokenizer.ipynb first"

In [ ]:
tok = ByteLevelBPETokenizer(f"{vocab_dir}/vocab.json", f"{vocab_dir}/merges.txt")
lines = load_lines(corpus_path)
device = "cuda" if torch.cuda.is_available() else "cpu"

model = TransformerAutoencoder(cfg).to(device)
print(f"model parameters: {model.num_parameters():,}  (device: {device})")
opt = torch.optim.AdamW(model.parameters(), lr=3e-4)
rng = random.Random(0)

In [ ]:
STEPS = 100
BATCH_SIZE = 4
losses = []

model.train()
for step in range(1, STEPS + 1):
    texts = [rng.choice(lines) for _ in range(BATCH_SIZE)]
    target_ids = encode_batch(tok, cfg, texts, cfg.max_seq_len).to(device)

    noisy_ids = target_ids.clone()
    for i in range(noisy_ids.size(0)):
        row = [t for t in target_ids[i].tolist() if t != cfg.pad_id]
        noised = corrupt(row, cfg, rng=rng)[: cfg.max_seq_len]
        noised = noised + [cfg.pad_id] * (cfg.max_seq_len - len(noised))
        noisy_ids[i] = torch.tensor(noised, dtype=torch.long, device=device)

    decoder_input_ids = shift_right(target_ids, cfg.bos_id)
    encoder_padding_mask = noisy_ids != cfg.pad_id

    logits = model(noisy_ids, decoder_input_ids, encoder_padding_mask)
    loss = F.cross_entropy(
        logits.view(-1, cfg.vocab_size), target_ids.view(-1), ignore_index=cfg.pad_id
    )

    opt.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    opt.step()
    losses.append(loss.item())

    if step % 10 == 0 or step == 1:
        print(f"step {step}/{STEPS}  loss {loss.item():.4f}")

In [ ]:
plt.plot(losses)
plt.xlabel("step")
plt.ylabel("cross-entropy loss")
plt.title("Denoising autoencoder training loss")
plt.show()

In [ ]:
ckpt_path = pathlib.Path("../checkpoints/model.pt")
ckpt_path.parent.mkdir(parents=True, exist_ok=True)
torch.save({"model_state_dict": model.state_dict(), "config": cfg}, ckpt_path)
print(f"saved checkpoint to {ckpt_path}")